# GEOG 592 — Module 4 Homework
Hailey Rodriguez

In [40]:
import csv

filename = "tabela1613.csv"

col_names = [
    "nivel",          # geographic level -- should be "MU" (município) for every real record
    "cod_municipio",  # IBGE municipality code -> this is the GIS join key
    "municipio",      # municipality name, with state abbreviation, e.g. "Belem (PA)"
    "total_valor",    # value for the "Total" (all crops) column -- not needed here
    "total_unidade",  # unit label that goes with total_valor
    "acai_valor",     # value for the Acai column -- the one we actually want
    "acai_unidade",   # unit label that goes with acai_valor
]

with open(filename, "r", encoding="utf-8-sig") as file:
    for _ in range(5):
        next(file)  # skip the 5 title / layered-header lines

    reader = csv.DictReader(file, fieldnames=col_names)
    rows = list(reader)

print("Records found:", len(rows))

Records found: 5563


## Part 2: Inspect the data

In [41]:
for row in rows[3]:
    print(row)

nivel
cod_municipio
municipio
total_valor
total_unidade
acai_valor
acai_unidade


In [42]:
for row in rows[-6:]:
    print(row)

{'nivel': '-', 'cod_municipio': 'Zero absoluto, não resultante de um cálculo ou arredondamento.\nEx: Em determinado município não existem pessoas de 14 anos de idade sem instrução.', 'municipio': None, 'total_valor': None, 'total_unidade': None, 'acai_valor': None, 'acai_unidade': None}
{'nivel': '0', 'cod_municipio': 'Zero resultante de um cálculo ou arredondamento.\nEx: A inflação do feijão em determinada Região Metropolitana foi 0.\nDeterminado município produziu 400 kg de sementes de girassol e os dados da tabela são expressos em toneladas.', 'municipio': None, 'total_valor': None, 'total_unidade': None, 'acai_valor': None, 'acai_unidade': None}
{'nivel': 'X', 'cod_municipio': 'Valor inibido para não identificar o informante.\nEx: Determinado município só possui uma empresa produtora de cimento, logo o valor de sua produção deve ser inibido.', 'municipio': None, 'total_valor': None, 'total_unidade': None, 'acai_valor': None, 'acai_unidade': None}
{'nivel': '..', 'cod_municipio': 'V

In [43]:
nivel_counts = {}

for row in rows:
    value = row["nivel"]
    nivel_counts[value] = nivel_counts.get(value, 0) + 1

# only print entries other than "MU" so we can see what the non-data rows look like
for key, count in nivel_counts.items():
    if key != "MU":
        print(repr(key), ":", count)

print()
print("Real 'MU' records:", nivel_counts.get("MU", 0))

'Fonte: IBGE - Produção Agrícola Municipal' : 1
'Notas' : 1
'1 - Os municípios sem informação para pelo menos um produto da lavoura permanente não aparecem nas listas.' : 1
'2 - A partir do ano de 2001 as quantidades produzidas dos produtos abacate, banana, caqui, figo, goiaba, laranja, limão, maçã, mamão, manga, maracujá, marmelo, pera, pêssego e tangerina passam a ser expressas em toneladas. Nos anos anteriores eram expressas em mil frutos, com exceção da banana, que era expressa em mil cachos. O rendimento médio passa a ser expresso em Kg/ha. Nos anos anteriores era expresso em frutos/ha, com exceção da banana, que era expressa em cachos/ha.' : 1
'3 - Veja em o\xa0documento AlteracoesUnidadesMedidaFrutas.pdf com as alterações de unidades de medida das frutíferas ocorridas em 2001 e a tabela de conversão fruto x quilograma.' : 1
'4 - Até 2001, café (em coco), a partir de 2002, café (beneficiado ou em grão).' : 1
'5 - A quantidade produzida de coco-da-baía é expressa em mil frutos e o

## Part 3: Keep only real municipality records

In [44]:
mu_rows = []

for row in rows:
    if row["nivel"] == "MU":
        mu_rows.append(row)

print("Before filtering:", len(rows), "rows")
print("After filtering: ", len(mu_rows), "rows")

Before filtering: 5563 rows
After filtering:  5541 rows


## Part 4: Filter to Pará (PA)

In [45]:
para_rows = []

for row in mu_rows:
    if row["cod_municipio"].startswith("15"):
        para_rows.append(row)

print("Pará municipalities found:", len(para_rows))

Pará municipalities found: 143


In [46]:
check_count = 0

for row in mu_rows:
    if row["municipio"].endswith("(PA)"):
        check_count += 1

print("Count by name suffix:", check_count)
print("Matches code-based filter:", check_count == len(para_rows))

Count by name suffix: 143
Matches code-based filter: True


## Part 5: Keep only the columns needed for the GIS join


In [47]:
selected_columns = []

for row in para_rows:
    selected_columns.append({
        "cod_municipio": row["cod_municipio"],
        "acai_valor": row["acai_valor"],
    })

for row in selected_columns[:5]:
    print(row)

{'cod_municipio': '1500107', 'acai_valor': '5600'}
{'cod_municipio': '1500131', 'acai_valor': '8000'}
{'cod_municipio': '1500206', 'acai_valor': '12000'}
{'cod_municipio': '1500305', 'acai_valor': '-'}
{'cod_municipio': '1500347', 'acai_valor': '10000'}


## Part 6: Expected data types


In [48]:
value = selected_columns[0]["acai_valor"]
print(value, "is datatype", type(value))

5600 is datatype <class 'str'>


## Part 7: Hunt for problems


In [49]:
def can_be_int(value):
    try:
        int(value)
        return True
    except ValueError:
        return False


def can_be_float(value):
    try:
        float(value)
        return True
    except ValueError:
        return False

In [50]:
error_found = False

for row in selected_columns:
    value = row["cod_municipio"]

    if value == "":
        print("Missing cod_municipio for a row")
        error_found = True
    elif not can_be_int(value):
        print("Suspicious cod_municipio:", value)
        error_found = True

if not error_found:
    print("Everything looks good.")

Everything looks good.


In [51]:
# also check for duplicate codes, which would break a GIS join later
seen = set()
duplicates_found = False

for row in selected_columns:
    code_value = row["cod_municipio"]
    if code_value in seen:
        print("Duplicate cod_municipio:", code_value)
        duplicates_found = True
    seen.add(code_value)

if not duplicates_found:
    print("No duplicate codes found.")

No duplicate codes found.


### Check `acai_valor`

IBGE uses special placeholder *symbols* instead of leaving cells blank:

- `"-"` means an actual, real **zero** (not a missing value).
- `"..."` means the value is genuinely **not available** (i.e. truly unknown/missing).

Both of those will fail a `can_be_float()` check right now, which is exactly what we want it
to catch.

In [52]:
error_found = False
placeholder_counts = {}

for row in selected_columns:
    value = row["acai_valor"]

    if not can_be_float(value):
        placeholder_counts[value] = placeholder_counts.get(value, 0) + 1
        error_found = True

if not error_found:
    print("All acai_valor entries already look numeric.")
else:
    print("Non-numeric values found in acai_valor:")
    for symbol, count in placeholder_counts.items():
        print(" ", repr(symbol), "->", count, "occurrences")

Non-numeric values found in acai_valor:
  '-' -> 21 occurrences
  '...' -> 4 occurrences


## Part 8: Clean the data


In [53]:
for row in selected_columns:
    if row["acai_valor"] == "-":
        row["acai_valor"] = "0"
    elif row["acai_valor"] == "...":
        row["acai_valor"] = ""

# re-run the same check from Part 7 to confirm the fix worked
error_found = False

for row in selected_columns:
    value = row["acai_valor"]
    if value != "" and not can_be_float(value):
        print("Still suspicious:", value)
        error_found = True

if not error_found:
    print("acai_valor is now clean: every entry is either numeric or a true empty/missing value.")

acai_valor is now clean: every entry is either numeric or a true empty/missing value.


In [54]:
missing_count = 0

for row in selected_columns:
    if row["acai_valor"] == "":
        missing_count += 1

print("Missing (unavailable) acai_valor entries:", missing_count)

Missing (unavailable) acai_valor entries: 4


In [55]:
selected_columns[:5]

[{'cod_municipio': '1500107', 'acai_valor': '5600'},
 {'cod_municipio': '1500131', 'acai_valor': '8000'},
 {'cod_municipio': '1500206', 'acai_valor': '12000'},
 {'cod_municipio': '1500305', 'acai_valor': '0'},
 {'cod_municipio': '1500347', 'acai_valor': '10000'}]

## Part 9: Export the cleaned CSV


In [56]:
fieldnames = selected_columns[0].keys()

with open("acai_para_clean.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(selected_columns)

print("Wrote", len(selected_columns), "rows to acai_para_clean.csv")

Wrote 143 rows to acai_para_clean.csv


In [57]:
# confirmation same pattern as Part 1
with open("acai_para_clean.csv", "r") as file:
    reader = csv.DictReader(file)
    check_rows = list(reader)

for row in check_rows[:5]:
    print(row)

{'cod_municipio': '1500107', 'acai_valor': '5600'}
{'cod_municipio': '1500131', 'acai_valor': '8000'}
{'cod_municipio': '1500206', 'acai_valor': '12000'}
{'cod_municipio': '1500305', 'acai_valor': '0'}
{'cod_municipio': '1500347', 'acai_valor': '10000'}
